# Lab 16 — Class-Balanced EfficientNet-B0
## 1. Experiment objective

Deepfake Detection for Video KYC.

Lab 14 (MobileNetV2) and Lab 15 (EfficientNet-B0) both reach ~90% test accuracy, but
**FAKE recall (~97-98%) is far higher than REAL recall (~52-56%)** because the training
set is roughly 1:5 REAL:FAKE (4,965 REAL vs 24,744 FAKE images). A model that is this
biased toward predicting FAKE will misclassify a large fraction of genuine users during
KYC onboarding — a costly failure mode in production, even though raw accuracy looks fine.

**Research question:** Does class-balanced training (inverse-class-frequency weighted
cross-entropy) improve REAL recall and overall balanced performance, without seriously
degrading FAKE detection?

**What changes vs. Lab 15:** only the training loss. Architecture (pretrained
EfficientNet-B0), image size (224x224), transforms, batch size (16), the identity-disjoint
train/val/test split, and the evaluation protocol are all identical to Lab 15, so any
difference in test-set metrics is attributable to the loss re-weighting alone.

**Lab 15 baseline reference (test set):** Accuracy 90.41% | FAKE F1 94.42% | REAL F1 66.00% |
ROC-AUC 0.9055.

The test set is evaluated **exactly once**, after model selection on validation data. It is
never used for training, augmentation, or checkpoint selection, and its class distribution
is left untouched.

## 2. Environment setup

Designed for a fresh Google Colab runtime with a T4 GPU. No Google Drive mounting is
required — the dataset is pulled from Kaggle and the checkpoint is saved under the
cloned repository's `outputs/` folder (download it manually before the runtime recycles;
see the appendix at the end of this notebook).

In [ ]:
!pip install -q kaggle scikit-learn

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path
from getpass import getpass

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

In [ ]:
# Clone (or reuse) the project repository so we have access to
# src.dataset.dataloader.FaceDataset and the rest of the project code.
REPO_URL = "https://github.com/Aishwarya-93/Deepfake-Detection-KYC.git"
REPO_NAME = "Deepfake-Detection-KYC"

if IN_COLAB:
    os.chdir("/content")
    if not Path(REPO_NAME).exists():
        exit_code = os.system(f"git clone {REPO_URL}")
        if exit_code != 0:
            raise RuntimeError("git clone failed. Check REPO_URL / network access.")
    os.chdir(REPO_NAME)
else:
    print("Not running in Colab - assuming the notebook already sits inside the repo.")

print("Working directory:", Path.cwd())

In [ ]:
def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing both
    `src/` and `data/` is found. Falls back to `start` if not found.
    Avoids hard-coding any machine-specific path."""
    for directory in [start] + list(start.parents):
        if (directory / "src").is_dir() and (directory / "data").is_dir():
            return directory
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

assert (PROJECT_ROOT / "src" / "dataset" / "dataloader.py").exists(), (
    "Could not locate src/dataset/dataloader.py under the detected project root. "
    "Check that the repository cloned correctly."
)

## 3. Imports

In [ ]:
from collections import Counter

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from src.dataset.dataloader import FaceDataset

print("PyTorch version:    ", torch.__version__)
print("Torchvision version: ", torchvision.__version__)

if torch.cuda.is_available():
    print("CUDA available: Yes")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA available: No (Runtime > Change runtime type > T4 GPU is recommended)")

## 4. Reproducibility / random seeds

`SEED = 42` matches `DEFAULT_RANDOM_STATE` used to build the identity-disjoint split in
`src/preprocessing/dataset_splitter.py`, so this notebook follows the same convention
already established in the project. Seeding Python's `random`, NumPy, and PyTorch (CPU +
CUDA) removes the main sources of run-to-run variance from weight initialization of the
new classifier head, data shuffling, and augmentation. `cudnn.deterministic = True` /
`cudnn.benchmark = False` trade a little GPU throughput for reproducible convolution
algorithms; exact bit-for-bit reproducibility across different GPU models is still not
guaranteed, but results should be closely reproducible on the same hardware/software
stack.

In [ ]:
import random

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Random seed set to:", SEED)

## 5. Kaggle authentication

Uses the `KAGGLE_API_TOKEN` Colab secret already used in Lab 14 / Lab 15. Falls back to
an interactive prompt only if that secret is not available.

In [ ]:
def load_kaggle_credentials():
    """Authenticate with Kaggle using the existing KAGGLE_API_TOKEN Colab
    secret. Falls back to an interactive prompt only if that secret isn't
    available."""

    token = None
    username = None

    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            token = None

    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        os.environ["KAGGLE_KEY"] = token

        if IN_COLAB:
            try:
                username = userdata.get("KAGGLE_USERNAME")
            except Exception:
                username = None
        if username:
            os.environ["KAGGLE_USERNAME"] = username

        print("Kaggle token loaded:", bool(token))
    else:
        print("KAGGLE_API_TOKEN secret not found.")
        print("Falling back to interactive input.")
        username = input("Kaggle username: ").strip()
        key = getpass("Kaggle API key: ").strip()

        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_json_path = kaggle_dir / "kaggle.json"
        kaggle_json_path.write_text(json.dumps({"username": username, "key": key}))
        kaggle_json_path.chmod(0o600)

        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key

        print("Kaggle credentials configured:", bool(username and key))


load_kaggle_credentials()

## 6. Dataset download

Same canonical dataset used by Lab 14/15 (`aishwarya99990/faceforensics-kyc-processed`),
downloaded to a single canonical location `PROJECT_ROOT/data/kaggle_processed` so a Colab
session that already ran Lab 14/15 can reuse the existing download instead of duplicating
it.

In [ ]:
KAGGLE_DATASET_SLUG = "aishwarya99990/faceforensics-kyc-processed"

# Single canonical dataset location - shared with Lab 14 / Lab 15, never duplicated.
DATASET_ROOT = PROJECT_ROOT / "data" / "kaggle_processed"
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_COUNTS = {
    "train": {"real": 4965, "fake": 24744},
    "val":   {"real": 1082, "fake": 5475},
    "test":  {"real": 1059, "fake": 5377},
}


def count_jpgs(folder: Path) -> int:
    return len(list(folder.glob("*.jpg"))) if folder.exists() else 0


def current_counts(root: Path):
    return {
        split: {label: count_jpgs(root / split / label) for label in ("real", "fake")}
        for split in ("train", "val", "test")
    }


counts_before = current_counts(DATASET_ROOT)
print("Existing counts:", counts_before)

complete_exists = all(
    counts_before[s][l] >= EXPECTED_COUNTS[s][l]
    for s in EXPECTED_COUNTS for l in EXPECTED_COUNTS[s]
)

if complete_exists:
    print("Complete dataset already present at:", DATASET_ROOT)
    print("Skipping download.")
else:
    print("Complete dataset not found. Downloading from Kaggle...")
    exit_code = os.system(
        f'kaggle datasets download -d "{KAGGLE_DATASET_SLUG}" -p "{DATASET_ROOT}" --unzip'
    )
    if exit_code != 0:
        raise RuntimeError(
            "kaggle datasets download failed. Check your Kaggle credentials "
            "(Section 5) and that you have accepted the dataset's terms on Kaggle."
        )

print("Dataset root:", DATASET_ROOT)

## 7. Dataset extraction

`kaggle datasets download --unzip` extracts directly into `DATASET_ROOT`, so this section
verifies the extraction produced the expected folder layout and scans every image for
zero-byte/corrupted files before any `Dataset`/`DataLoader` is built — a bad file must
never reach a DataLoader worker mid-training.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
QUARANTINE_ROOT = DATASET_ROOT.parent / "kaggle_processed_quarantine"


def verify_structure(root: Path):
    missing = []
    for split in ("train", "val", "test"):
        for label in ("real", "fake"):
            folder = root / split / label
            if not folder.exists():
                missing.append(str(folder))
    if missing:
        raise RuntimeError("Missing dataset folders:\n" + "\n".join(missing))


def find_bad_images(root: Path):
    """Return a list of (path, reason) for every zero-byte or unreadable image."""
    bad = []
    for split in ("train", "val", "test"):
        for label in ("real", "fake"):
            folder = root / split / label
            if not folder.exists():
                continue
            for path in folder.iterdir():
                if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
                    continue
                try:
                    if path.stat().st_size == 0:
                        bad.append((path, "zero-byte file"))
                        continue
                    with Image.open(path) as img:
                        img.verify()
                    with Image.open(path) as img:
                        img.convert("RGB").load()
                except Exception as e:
                    bad.append((path, repr(e)))
    return bad


def quarantine_bad_images(bad_files):
    for path, reason in bad_files:
        rel = path.relative_to(DATASET_ROOT)
        dest = QUARANTINE_ROOT / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(dest))
    print(f"Quarantined {len(bad_files)} file(s) to: {QUARANTINE_ROOT}")


from PIL import Image

verify_structure(DATASET_ROOT)
print("Folder structure OK.")

print("Scanning for corrupted/unreadable images (this can take a few minutes)...")
bad_files = find_bad_images(DATASET_ROOT)
print(f"Found {len(bad_files)} corrupted/unreadable/zero-byte image(s).")

for path, reason in bad_files[:50]:
    print(" -", path, "|", reason)
if len(bad_files) > 50:
    print(f"  ... and {len(bad_files) - 50} more")

if bad_files:
    quarantine_bad_images(bad_files)

remaining_bad = find_bad_images(DATASET_ROOT)
print("Remaining corrupted images:", len(remaining_bad))

if remaining_bad:
    raise RuntimeError(
        f"{len(remaining_bad)} corrupted image(s) still present after cleanup. "
        "Inspect QUARANTINE_ROOT / re-run this cell before proceeding."
    )

## 8. Dataset path verification

In [ ]:
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

for name, path in [("TRAIN_DIR", TRAIN_DIR), ("VAL_DIR", VAL_DIR), ("TEST_DIR", TEST_DIR)]:
    print(f"{name}: {path} (exists={path.exists()})")
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

post_cleanup_counts = current_counts(DATASET_ROOT)

for split in EXPECTED_COUNTS:
    for label in EXPECTED_COUNTS[split]:
        actual = post_cleanup_counts[split][label]
        expected = EXPECTED_COUNTS[split][label]
        if actual < expected:
            raise RuntimeError(
                f"{split}/{label} has {actual} images, but {expected} were expected. "
                "The dataset appears incomplete; stopping before training."
            )

print("\nDataset paths verified against the expected complete counts.")

## 9. Dataset statistics

In [ ]:
print("DATASET STATISTICS (post-cleanup, on disk)")
total_by_split = {}
for split, labels in post_cleanup_counts.items():
    total = labels["real"] + labels["fake"]
    total_by_split[split] = total
    ratio = labels["fake"] / labels["real"] if labels["real"] else float("inf")
    print(
        f"{split.upper():5s} | real: {labels['real']:6d} | fake: {labels['fake']:6d} "
        f"| total: {total:6d} | fake:real ratio = {ratio:.2f}:1"
    )

print("\nExpected complete totals: TRAIN=29709, VAL=6557, TEST=6436")

if any(v == 0 for v in total_by_split.values()):
    raise RuntimeError("One or more dataset splits are empty.")

## 10. Transform definitions

**Unchanged from Lab 15** — same 224x224 resize, the same light train-time augmentation
(horizontal flip + +/-10 degree rotation), and the same ImageNet normalization statistics
for the pretrained EfficientNet-B0 backbone. No augmentation is applied to validation or
test data. Keeping preprocessing identical to Lab 15 isolates the loss re-weighting as the
only training-strategy change under test.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("train_transform:", train_transform)
print("eval_transform: ", eval_transform)

## 11. Dataset / DataLoader creation

`BATCH_SIZE = 16` matches Lab 14/15 for a fair comparison. `NUM_WORKERS` is left
configurable so it can be tuned for whatever machine actually runs training (Colab, a
local GPU box, etc.) without touching any other cell.

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2  # configurable: safe default for Colab; raise on a machine with more CPU cores

train_dataset = FaceDataset(str(TRAIN_DIR), transform=train_transform)
val_dataset = FaceDataset(str(VAL_DIR), transform=eval_transform)
test_dataset = FaceDataset(str(TEST_DIR), transform=eval_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)

print("TRAIN dataset:", len(train_dataset))
print("TRAIN loader:", len(train_loader))
print("VAL dataset:", len(val_dataset))
print("VAL loader:", len(val_loader))
print("TEST dataset:", len(test_dataset))
print("TEST loader:", len(test_loader))

if len(train_dataset) != total_by_split["train"]:
    raise RuntimeError("FaceDataset train count does not match filesystem count.")
if len(val_dataset) != total_by_split["val"]:
    raise RuntimeError("FaceDataset val count does not match filesystem count.")
if len(test_dataset) != total_by_split["test"]:
    raise RuntimeError("FaceDataset test count does not match filesystem count.")

## 12. Class distribution calculation

Computed directly from `train_dataset.labels` — the exact label list the `DataLoader`
will consume — rather than re-globbing the filesystem, so the class weights below are
guaranteed to correspond to the real training-time class frequencies (labels: `0 = REAL`,
`1 = FAKE`, matching `FaceDataset`).

In [ ]:
train_class_counts = Counter(train_dataset.labels)
NUM_REAL = train_class_counts[0]
NUM_FAKE = train_class_counts[1]
NUM_TRAIN_SAMPLES = NUM_REAL + NUM_FAKE
NUM_CLASSES = 2

print("Training label distribution (0=REAL, 1=FAKE):", dict(train_class_counts))
print("NUM_REAL:", NUM_REAL)
print("NUM_FAKE:", NUM_FAKE)
print("NUM_TRAIN_SAMPLES:", NUM_TRAIN_SAMPLES)
print(f"Imbalance ratio (FAKE:REAL): {NUM_FAKE / NUM_REAL:.2f} : 1")

assert NUM_REAL + NUM_FAKE == len(train_dataset), "Class counts must sum to the full training set."
assert NUM_REAL == EXPECTED_COUNTS["train"]["real"], "REAL count does not match the verified dataset."
assert NUM_FAKE == EXPECTED_COUNTS["train"]["fake"], "FAKE count does not match the verified dataset."


## 13. Class-weight calculation

Uses **inverse class frequency** weighting:

`weight_c = N / (num_classes * count_c)`

This is a standard, well-justified, stable class-balancing scheme for cross-entropy loss
(it is exactly what scikit-learn's `class_weight="balanced"` computes). The minority class
(REAL) receives a proportionally larger weight so misclassifying a REAL image costs the
optimizer more than misclassifying a FAKE image, without touching the data itself (no
oversampling/duplication of images, no change to validation/test).

In [ ]:
class_weights = torch.tensor(
    [
        NUM_TRAIN_SAMPLES / (NUM_CLASSES * NUM_REAL),  # REAL (class 0)
        NUM_TRAIN_SAMPLES / (NUM_CLASSES * NUM_FAKE),  # FAKE (class 1)
    ],
    dtype=torch.float32,
)

print("Class weights [REAL, FAKE]:", class_weights.tolist())
print(f"  REAL weight: {class_weights[0]:.4f}")
print(f"  FAKE weight: {class_weights[1]:.4f}")
print(f"  Ratio REAL/FAKE weight: {(class_weights[0] / class_weights[1]):.2f}")

assert class_weights[0] > class_weights[1], (
    "REAL is the minority class in the training set and must receive the larger weight."
)

## 14. EfficientNet-B0 model setup

Identical to Lab 15: pretrained ImageNet weights, classifier head replaced with a 2-way
linear layer for REAL (0) vs FAKE (1).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace the final classifier for REAL (0) vs FAKE (1).
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

model = model.to(device)
print(model.classifier)

In [ ]:
# Forward-pass sanity check: output dimension must be [batch, 2].
images, labels = next(iter(train_loader))
images = images.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
assert outputs.shape[1] == 2, "Model output dimension must be 2 (REAL, FAKE)."
model.train()

## 15. Class-balanced loss setup

The **only** intentional training-strategy change vs. Lab 15: `CrossEntropyLoss` is given
the inverse-frequency `class_weights` computed above, so training gradients are
class-balanced. A second, unweighted `CrossEntropyLoss` is kept for computing
validation/test loss values that stay directly comparable in scale to Lab 14/15's
history logs (the class weight only rescales the training loss used for backprop — it
does not change argmax predictions, so accuracy/precision/recall/F1/ROC-AUC are
unaffected either way).

In [ ]:
train_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
eval_criterion = nn.CrossEntropyLoss()  # unweighted - keeps val/test loss comparable to Lab 15

print("Training criterion (class-weighted):", train_criterion)
print("Evaluation criterion (unweighted):   ", eval_criterion)

## 16. Optimizer / scheduler

Same `Adam(lr=1e-4)` optimizer as Lab 15, with **no learning-rate scheduler** — keeping
every other hyperparameter identical to Lab 15 isolates the class-weighted loss as the
sole training-strategy change being evaluated.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = None  # intentionally none - see markdown above

print("Optimizer:", optimizer.__class__.__name__)
print("Scheduler:", scheduler)

## 17. Training loop

`train_one_epoch` optimizes the **class-weighted** loss. `evaluate` is a generic pass that
accepts whichever criterion is passed in, so it is reused both for validation (with the
unweighted `eval_criterion`) and, later, for the test set.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    if total == 0:
        raise RuntimeError("Training loader produced zero samples.")

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise RuntimeError("Evaluation loader produced zero samples.")

    return running_loss / total, correct / total

print("train_one_epoch and evaluate defined.")

## 18. Validation after every epoch

The model is validated on the **complete, untouched** validation split after every
training epoch, using the unweighted `eval_criterion` so `val_loss` is on the same scale
as Lab 14/15's logged validation loss.

## 19. Best-model checkpointing

The checkpoint is selected by **validation accuracy**, the same selection criterion used
in Lab 14/15 — this keeps model selection identical across labs so any change in test
metrics reflects the class-weighted loss, not a change in the evaluation/selection
protocol. Only training and validation data are used here; the test set is never touched
during this loop.

In [ ]:
EPOCHS = 5

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = OUTPUTS_DIR / "models"
PLOTS_DIR = OUTPUTS_DIR / "plots"
RESULTS_DIR = OUTPUTS_DIR / "results"

for d in (MODELS_DIR, PLOTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODELS_DIR / "efficientnetb0_classbalanced_best.pth"
print("Checkpoint will be saved to:", BEST_MODEL_PATH)

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

best_val_accuracy = 0.0

print("Training on complete training dataset:", len(train_dataset), "images")
print("Validating on complete validation dataset:", len(val_dataset), "images")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, train_criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, eval_criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss (weighted): {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}% | "
        f"Val Loss (unweighted): {val_loss:.4f} | Val Acc: {val_acc * 100:.2f}%"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("  \u2713 Best model saved")

with open(RESULTS_DIR / "lab16_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nBest validation accuracy:", f"{best_val_accuracy * 100:.2f}%")
print("Checkpoint:", BEST_MODEL_PATH)
print("Checkpoint exists:", BEST_MODEL_PATH.exists())

if not BEST_MODEL_PATH.exists():
    raise RuntimeError(
        "Checkpoint file was not found after training. "
        "Training may have failed before any epoch improved on val accuracy."
    )

## 20. Training curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss (weighted)")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss (unweighted)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Class-Balanced EfficientNet-B0 - Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, [a * 100 for a in history["train_accuracy"]], marker="o", label="Train Acc")
axes[1].plot(epochs_range, [a * 100 for a in history["val_accuracy"]], marker="o", label="Val Acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Class-Balanced EfficientNet-B0 - Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

curves_path = PLOTS_DIR / "efficientnetb0_classbalanced_training_curves.png"
plt.savefig(curves_path, dpi=150)
plt.show()

print("Saved training curves to:", curves_path)

## 21. Test evaluation

Loads the best checkpoint (selected on validation accuracy, Section 19) into a fresh
model skeleton and runs it **once** over the complete, untouched test split. Both overall
accuracy and per-class (REAL/FAKE) precision, recall, and F1 are computed, plus ROC-AUC
using the FAKE-class probability as the positive-class score — matching Lab 15's
evaluation protocol exactly.

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_MODEL_PATH}")

eval_model = efficientnet_b0(weights=None)
num_features = eval_model.classifier[1].in_features
eval_model.classifier[1] = nn.Linear(num_features, 2)

eval_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
eval_model = eval_model.to(device)
eval_model.eval()

print("Loaded best checkpoint from:", BEST_MODEL_PATH)

In [ ]:
test_loss, _ = evaluate(eval_model, test_loader, eval_criterion, device)

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        outputs = eval_model(images)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities[:, 1].cpu().numpy())

if len(all_labels) != len(test_dataset):
    raise RuntimeError(
        f"Only evaluated {len(all_labels)} test images; expected {len(test_dataset)}."
    )

test_accuracy = accuracy_score(all_labels, all_predictions)

# Per-class precision/recall/F1 for REAL (0) and FAKE (1).
precisions, recalls, f1s, supports = precision_recall_fscore_support(
    all_labels, all_predictions, labels=[0, 1], zero_division=0
)
test_precision_real, test_precision_fake = precisions
test_recall_real, test_recall_fake = recalls
test_f1_real, test_f1_fake = f1s
test_support_real, test_support_fake = supports

test_roc_auc = roc_auc_score(all_labels, all_probabilities)

print("===================================")
print("CLASS-BALANCED EFFICIENTNET-B0 TEST RESULTS")
print("===================================")
print(f"Test samples evaluated: {len(all_labels)} (expected {len(test_dataset)})")
print(f"Test Loss (unweighted) : {test_loss:.4f}")
print(f"Accuracy               : {test_accuracy * 100:.2f}%")
print()
print(f"REAL  | precision: {test_precision_real*100:.2f}% | recall: {test_recall_real*100:.2f}% | "
      f"F1: {test_f1_real*100:.2f}% | support: {test_support_real}")
print(f"FAKE  | precision: {test_precision_fake*100:.2f}% | recall: {test_recall_fake*100:.2f}% | "
      f"F1: {test_f1_fake*100:.2f}% | support: {test_support_fake}")
print()
print(f"ROC-AUC: {test_roc_auc:.4f}")

## 22. Confusion matrix

Labels: `0 = REAL`, `1 = FAKE` (matches `FaceDataset`'s class mapping). Rows are true
labels, columns are predicted labels.

In [ ]:
cm = confusion_matrix(all_labels, all_predictions)
class_names = ["REAL", "FAKE"]

print("Confusion Matrix (rows=true, cols=predicted):")
print(cm)

tn, fp, fn, tp = cm.ravel()
num_correct = tn + tp
num_incorrect = fp + fn
print(f"\nCorrect predictions:   {num_correct}")
print(f"Incorrect predictions: {num_incorrect}")
print(f"  REAL misclassified as FAKE (FP): {fp}")
print(f"  FAKE misclassified as REAL (FN): {fn}")

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix - Class-Balanced EfficientNet-B0")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()

cm_path = PLOTS_DIR / "efficientnetb0_classbalanced_confusion_matrix.png"
plt.savefig(cm_path, dpi=150)
plt.show()

print("Saved confusion matrix plot to:", cm_path)

## 23. Classification report

In [ ]:
report_text = classification_report(
    all_labels, all_predictions, target_names=class_names, zero_division=0
)
print(report_text)

report_dict = classification_report(
    all_labels, all_predictions, target_names=class_names,
    zero_division=0, output_dict=True,
)

## 24. ROC-AUC

Computed once already in Section 21 (`test_roc_auc`), restated here for clarity since the
project's results checklist calls it out as its own item. Uses the FAKE-class softmax
probability as the positive-class score, exactly as in Lab 14/15.

In [ ]:
print(f"Test ROC-AUC (positive class = FAKE): {test_roc_auc:.4f}")

## 25. Comparison against Lab 15 (and Lab 14)

Reference numbers below are copied from the completed Lab 14 and Lab 15 notebooks/results
(not recomputed here) and are not modified by this notebook.

In [ ]:
mobilenetv2_baseline = {
    "model": "MobileNetV2 (Lab 14)",
    "test_accuracy": 0.9024,
    "test_precision_real": 0.82, "test_recall_real": 0.52, "test_f1_real": 0.64,
    "test_precision_fake": 0.9120, "test_recall_fake": 0.9775, "test_f1_fake": 0.9436,
    "test_roc_auc": 0.9020,
}

efficientnetb0_baseline = {
    "model": "EfficientNet-B0 baseline (Lab 15)",
    "test_accuracy": 0.9041,
    "test_precision_real": 0.80, "test_recall_real": 0.56, "test_f1_real": 0.66,
    "test_precision_fake": 0.9183, "test_recall_fake": 0.9717, "test_f1_fake": 0.9442,
    "test_roc_auc": 0.9055,
}

classbalanced_result = {
    "model": "EfficientNet-B0 class-balanced (Lab 16)",
    "test_accuracy": test_accuracy,
    "test_precision_real": test_precision_real, "test_recall_real": test_recall_real, "test_f1_real": test_f1_real,
    "test_precision_fake": test_precision_fake, "test_recall_fake": test_recall_fake, "test_f1_fake": test_f1_fake,
    "test_roc_auc": test_roc_auc,
}

print("MODEL COMPARISON")
print("=" * 100)
header = f"{'Model':<38}{'Acc':>8}{'REAL P':>8}{'REAL R':>8}{'REAL F1':>9}{'FAKE P':>8}{'FAKE R':>8}{'FAKE F1':>9}{'ROC-AUC':>9}"
print(header)
for row in (mobilenetv2_baseline, efficientnetb0_baseline, classbalanced_result):
    print(
        f"{row['model']:<38}"
        f"{row['test_accuracy']*100:7.2f}%"
        f"{row['test_precision_real']*100:7.2f}%"
        f"{row['test_recall_real']*100:7.2f}%"
        f"{row['test_f1_real']*100:8.2f}%"
        f"{row['test_precision_fake']*100:7.2f}%"
        f"{row['test_recall_fake']*100:7.2f}%"
        f"{row['test_f1_fake']*100:8.2f}%"
        f"{row['test_roc_auc']:9.4f}"
    )

real_recall_delta = classbalanced_result["test_recall_real"] - efficientnetb0_baseline["test_recall_real"]
fake_f1_delta = classbalanced_result["test_f1_fake"] - efficientnetb0_baseline["test_f1_fake"]
accuracy_delta = classbalanced_result["test_accuracy"] - efficientnetb0_baseline["test_accuracy"]
roc_auc_delta = classbalanced_result["test_roc_auc"] - efficientnetb0_baseline["test_roc_auc"]

print("\nDeltas vs. Lab 15 EfficientNet-B0 baseline:")
print(f"  REAL recall : {real_recall_delta:+.4f} ({real_recall_delta*100:+.2f} pp)")
print(f"  FAKE F1     : {fake_f1_delta:+.4f} ({fake_f1_delta*100:+.2f} pp)")
print(f"  Accuracy    : {accuracy_delta:+.4f} ({accuracy_delta*100:+.2f} pp)")
print(f"  ROC-AUC     : {roc_auc_delta:+.4f}")

FAKE_F1_DEGRADATION_TOLERANCE = 0.05  # pp of FAKE F1 - a judgment call, not a hard requirement

print("\nResearch question: does class balancing improve REAL recall without seriously")
print("degrading FAKE detection?")
if real_recall_delta > 0 and fake_f1_delta > -FAKE_F1_DEGRADATION_TOLERANCE:
    print(
        f"  -> YES: REAL recall improved by {real_recall_delta*100:.2f} pp while FAKE F1 "
        f"changed by {fake_f1_delta*100:+.2f} pp (within the {FAKE_F1_DEGRADATION_TOLERANCE*100:.0f} pp tolerance)."
    )
elif real_recall_delta > 0:
    print(
        f"  -> PARTIAL: REAL recall improved by {real_recall_delta*100:.2f} pp, but FAKE F1 "
        f"dropped by {-fake_f1_delta*100:.2f} pp, exceeding the {FAKE_F1_DEGRADATION_TOLERANCE*100:.0f} pp tolerance."
    )
else:
    print("  -> NO: class-weighted training did not improve REAL recall over the Lab 15 baseline.")

## 26. Save results

Full results are saved as JSON, plus a flat one-row CSV summary for quick cross-lab
comparison. This does not touch Lab 14/15's own result files.

In [ ]:
import csv

results_summary = {
    "model": "EfficientNet-B0 (class-balanced)",
    "seed": SEED,
    "training_config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "image_size": 224,
        "optimizer": "Adam",
        "learning_rate": 1e-4,
        "scheduler": None,
        "loss": "CrossEntropyLoss (train: class-weighted, eval: unweighted)",
        "pretrained": True,
    },
    "class_weights": {"real": float(class_weights[0]), "fake": float(class_weights[1])},
    "train_class_counts": {"real": NUM_REAL, "fake": NUM_FAKE},
    "dataset_counts": post_cleanup_counts,
    "history": {
        "train_loss": history["train_loss"],
        "train_accuracy": history["train_accuracy"],
        "val_loss": history["val_loss"],
        "val_accuracy": history["val_accuracy"],
    },
    "best_val_accuracy": best_val_accuracy,
    "test_samples_evaluated": len(all_labels),
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "test_precision_real": test_precision_real,
    "test_recall_real": test_recall_real,
    "test_f1_real": test_f1_real,
    "test_precision_fake": test_precision_fake,
    "test_recall_fake": test_recall_fake,
    "test_f1_fake": test_f1_fake,
    "test_roc_auc": test_roc_auc,
    "confusion_matrix": cm.tolist(),
    "correct_predictions": int(num_correct),
    "incorrect_predictions": int(num_incorrect),
    "classification_report": report_dict,
    "checkpoint_path": str(BEST_MODEL_PATH),
    "comparison": {
        "mobilenetv2_baseline_lab14": mobilenetv2_baseline,
        "efficientnetb0_baseline_lab15": efficientnetb0_baseline,
        "efficientnetb0_classbalanced_lab16": classbalanced_result,
        "real_recall_delta_vs_lab15": real_recall_delta,
        "fake_f1_delta_vs_lab15": fake_f1_delta,
    },
}

results_json_path = RESULTS_DIR / "lab16_classbalanced_efficientnet_results.json"
with open(results_json_path, "w") as f:
    json.dump(results_summary, f, indent=2)

results_csv_path = RESULTS_DIR / "lab16_classbalanced_efficientnet_results.csv"
csv_row = {
    "model": results_summary["model"],
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_val_accuracy": best_val_accuracy,
    "test_accuracy": test_accuracy,
    "test_precision_real": test_precision_real,
    "test_recall_real": test_recall_real,
    "test_f1_real": test_f1_real,
    "test_precision_fake": test_precision_fake,
    "test_recall_fake": test_recall_fake,
    "test_f1_fake": test_f1_fake,
    "test_roc_auc": test_roc_auc,
    "class_weight_real": float(class_weights[0]),
    "class_weight_fake": float(class_weights[1]),
    "checkpoint_path": str(BEST_MODEL_PATH),
}
with open(results_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(csv_row.keys()))
    writer.writeheader()
    writer.writerow(csv_row)

print("Saved results JSON to:", results_json_path)
print("Saved results CSV to: ", results_csv_path)

## 27. Final experiment summary

In [ ]:
print("===================================")
print("FINAL LAB 16 SUMMARY - CLASS-BALANCED EFFICIENTNET-B0")
print("===================================")
print(f"Seed                 : {SEED}")
print(f"Class weights [R,F]  : {class_weights.tolist()}")
print(f"Best Val Accuracy    : {best_val_accuracy * 100:.2f}%")
print(f"Test Accuracy        : {test_accuracy * 100:.2f}%")
print(f"Test REAL  P/R/F1    : {test_precision_real*100:.2f}% / {test_recall_real*100:.2f}% / {test_f1_real*100:.2f}%")
print(f"Test FAKE  P/R/F1    : {test_precision_fake*100:.2f}% / {test_recall_fake*100:.2f}% / {test_f1_fake*100:.2f}%")
print(f"Test ROC-AUC         : {test_roc_auc:.4f}")
print()
print(f"vs. Lab 15 baseline  : REAL recall {real_recall_delta:+.4f} | FAKE F1 {fake_f1_delta:+.4f} | "
      f"Accuracy {accuracy_delta:+.4f} | ROC-AUC {roc_auc_delta:+.4f}")
print()
print("Checkpoint exists    :", BEST_MODEL_PATH.exists())
print("Checkpoint path      :", BEST_MODEL_PATH)
print("Results JSON         :", results_json_path)
print("Results CSV          :", results_csv_path)
print()
print("Test set was evaluated exactly once, after checkpoint selection on validation")
print("accuracy. Test distribution, split, and identities were not modified.")
print()
print("Next step: external evaluation on the held-out DeepFakeDetection (DFD) set.")

## Appendix: Download the checkpoint

`/content` is ephemeral and disconnecting the Colab runtime deletes everything saved
above. Run this cell **manually** whenever you want to download the best checkpoint to
your local machine (it does not run automatically, and this repository's `.gitignore`
already excludes `*.pth` files from being committed).

In [ ]:
# Run this cell manually to download the checkpoint - it does not run automatically.
if IN_COLAB:
    from google.colab import files
    if BEST_MODEL_PATH.exists():
        files.download(str(BEST_MODEL_PATH))
    else:
        print("Checkpoint not found at:", BEST_MODEL_PATH)
else:
    print("Not running in Colab - checkpoint is already available locally at:")
    print(BEST_MODEL_PATH)

## Expected workflow

1. Select **T4 GPU** in Colab (Runtime > Change runtime type).
2. Runtime > Restart session, then Run all.
3. Confirm Section 9's dataset statistics match the expected complete counts.
4. Confirm Section 13's class weights: REAL weight > FAKE weight.
5. Let all 5 epochs finish (Sections 17-19).
6. Record the Section 21/25 test metrics and the Section 25 comparison verdict.
7. Run the appendix cell and let `files.download(...)` save the checkpoint locally.
8. Compare against the Lab 15 EfficientNet-B0 baseline (90.41% accuracy, 0.9055 ROC-AUC)
   and the Lab 14 MobileNetV2 baseline (90.24% accuracy, 0.9020 ROC-AUC).

**Lab 15 baseline reference:** 90.41% test accuracy, REAL F1 66.00%, FAKE F1 94.42%,
ROC-AUC 0.9055.